# 패키지 import

In [1]:
### 필요 패키지 import ###
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import numpy as np
import pandas as pd
import PIL
import matplotlib.pyplot as plt
import os
from tqdm import tqdm, notebook

In [2]:
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

print(torch.__version__)

1.10.1


# Data Load

In [3]:
seed = 7

classes_names = ["포트홀 없음", "포트홀", "보수 완료된 포트홀"]
train_data = pd.read_csv("Train_datasets_labels.csv")
valid_data = pd.read_csv("Validation_datasets_labels.csv")


train_data = train_data.sample(frac=1, random_state=seed).reset_index(drop=True)
valid_data = valid_data.sample(frac=1, random_state=seed).reset_index(drop=True)

print(len(train_data))
print(len(valid_data))

23389
300


# Define Data-Sets

In [4]:
class MyDataset(Dataset):
    def __init__(self, df):
        self.df = df
        self.x=0; self.y=0
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        file_path = self.df.iloc[idx][0]
        img = np.array(PIL.Image.open(train_data.iloc[idx][0])) / 255.
        label = torch.tensor(train_data.iloc[idx][1])
        
        self.x = torch.tensor(img).float()
        self.y = F.one_hot(label, num_classes=3).float()
        
        return self.x, self.y
    
train = MyDataset(train_data)
train_loader = DataLoader(train, batch_size = 16)
valid = MyDataset(valid_data)
valid_loader = DataLoader(valid, batch_size = 16)

# Define Train,Valid Function

In [19]:
loss_fn = nn.CrossEntropyLoss()

def calc_acc(X, Y):
    x_val, x_idx = torch.max(X, dim=1)
    y_val, y_idx = torch.max(Y, dim=1)
    return (x_idx == y_idx).sum().item()

def train(EPOCHS, model, train_loader, opt):
    for epoch in range(1, EPOCHS+1):
        model.train()
        train_acc = 0
        print("### EPOCH {} ###".format(epoch))
        for batch_idx, (img,label) in enumerate(notebook.tqdm(train_loader)):
            img, label = img.to(DEVICE), label.to(DEVICE)
            
            output = model(img)
            loss = loss_fn(output, label)
            
            opt.zero_grad()
            loss.backward()
            opt.step()
            
            train_acc += calc_acc(output, label)
            if batch_idx % 100 == 0 and batch_idx != 0:
                print("Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}\t Acc : {:.3f}".format(
                    epoch, 
                    batch_idx * len(img), 
                    len(train_loader.dataset), 
                    100. * batch_idx / len(train_loader), 
                    loss.item(),
                    train_acc / len(train_loader.dataset)))
        t_loss, t_acc = evaluate(model, valid_loader)
        print("[{}] Test Loss: {:.4f}\t accuracy: {:.2f}%\n".format(epoch, t_loss, t_acc))
                
def evaluate(model, valid_loader):
    model.eval()
    t_loss = 0
    correct = 0
    
    with torch.no_grad():
        for img, label in notebook.tqdm(valid_loader):
            img, label = img.to(DEVICE), label.to(DEVICE)
            
            output = model(img)
            t_loss += loss_fn(output, label)
            
#             print(output.shape)
#             print(label.shape)
            correct += calc_acc(output, label)
    
    print(torch.max(output, dim=1)[1])
    print(torch.max(label, dim=1)[1])
    t_loss /= len(valid_loader)
    t_acc = 100. * correct / len(valid_loader.dataset)
    return t_loss, t_acc

# Define Model

In [6]:
USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")

print(USE_CUDA)
print(DEVICE)

True
cuda


In [7]:
class Model1(nn.Module):
    def __init__(self):
        super(Model1, self).__init__()
        self.conv1 = nn.Conv2d(in_channels = 3, out_channels = 8, kernel_size = 3, padding = 1)
        self.conv2 = nn.Conv2d(in_channels = 8, out_channels = 16, kernel_size = 3, padding = 1)
        self.conv3 = nn.Conv2d(in_channels = 16, out_channels = 32, kernel_size = 3, padding = 1)
        self.conv4 = nn.Conv2d(in_channels = 32, out_channels = 64, kernel_size = 3, padding = 1)
        self.pool = nn.MaxPool2d(kernel_size = 2, stride = 2)
        self.fc1 = nn.Linear(20 * 20 * 64, 1024)
        self.fc2 = nn.Linear(1024, 64)
        self.fc3 = nn.Linear(64, 3)
        
        self.conv1_bn = nn.BatchNorm2d(8)
        self.conv2_bn = nn.BatchNorm2d(16)
        self.conv3_bn = nn.BatchNorm2d(32)
        self.conv4_bn = nn.BatchNorm2d(64)
        
        self.act_fn = nn.ReLU()
        
        self.dropout_p = 0.2            # drop out 비율 값
        
    def forward(self, x):
        x = x.permute(0,3,1,2)
        x = self.conv1(x) # 320 * 320 * 3 -> 320 * 320 * 8
        x = self.conv1_bn(x)
        x = self.act_fn(x)     # 320 * 320 * 8
        x = self.pool(x)  # 160 * 160 * 8
        
        x = self.conv2(x) # 160 * 160 * 8 -> 160 * 160 * 16
        x = self.conv2_bn(x)
        x = self.act_fn(x)     # 160 * 160 * 16
        x = self.pool(x)  # 80  *  80 * 16
        
        x = self.conv3(x) # 80 * 80 * 16 -> 80 * 80 * 32
        x = self.conv3_bn(x)
        x = self.act_fn(x)     # 80 * 80 * 32
        x = self.pool(x)  # 40 * 40 * 32
        
        x = self.conv4(x) # 40 * 40 * 32 -> 40* 40 * 64
        x = self.conv4_bn(x)
        x = self.act_fn(x)     # 40 * 40 * 64
        x = self.pool(x)  # 20 * 20 * 64
        
        x = x.contiguous().view(-1, 20*20*64)
        x = self.fc1(x)
        x = F.dropout(x, p = self.dropout_p) # Drop out 적용
        x = self.act_fn(x)
        x = self.fc2(x)
        x = F.dropout(x, p = self.dropout_p) # Drop out 적용
        x = self.act_fn(x)
        x = self.fc3(x)
        x = F.log_softmax(x, dim=1)
        return x

In [8]:
model = Model1().to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr = 0.001)
print("DEVICE: ", DEVICE)
print("MODEL: ", model)

DEVICE:  cuda
MODEL:  Model1(
  (conv1): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=25600, out_features=1024, bias=True)
  (fc2): Linear(in_features=1024, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=3, bias=True)
  (conv1_bn): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2_bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3_bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv4_bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (act_fn): ReLU()


In [9]:
train(15, model, train_loader, optimizer)

### EPOCH 1 ###


  0%|          | 0/1462 [00:00<?, ?it/s]

Train Epoch: 1 [1600/23389 (7%)]	Loss: 1.112280	 Acc : 0.032
Train Epoch: 1 [3200/23389 (14%)]	Loss: 0.827747	 Acc : 0.068
Train Epoch: 1 [4800/23389 (21%)]	Loss: 0.436387	 Acc : 0.108
Train Epoch: 1 [6400/23389 (27%)]	Loss: 0.869975	 Acc : 0.148
Train Epoch: 1 [8000/23389 (34%)]	Loss: 0.655661	 Acc : 0.191
Train Epoch: 1 [9600/23389 (41%)]	Loss: 0.573134	 Acc : 0.234
Train Epoch: 1 [11200/23389 (48%)]	Loss: 0.485414	 Acc : 0.277
Train Epoch: 1 [12800/23389 (55%)]	Loss: 0.826841	 Acc : 0.322
Train Epoch: 1 [14400/23389 (62%)]	Loss: 0.607358	 Acc : 0.366
Train Epoch: 1 [16000/23389 (68%)]	Loss: 0.695722	 Acc : 0.411
Train Epoch: 1 [17600/23389 (75%)]	Loss: 0.778819	 Acc : 0.457
Train Epoch: 1 [19200/23389 (82%)]	Loss: 0.786855	 Acc : 0.504
Train Epoch: 1 [20800/23389 (89%)]	Loss: 0.631496	 Acc : 0.548
Train Epoch: 1 [22400/23389 (96%)]	Loss: 0.675458	 Acc : 0.594


  0%|          | 0/19 [00:00<?, ?it/s]

[1] Test Loss: 0.6592	 accuracy: 60.67%

### EPOCH 2 ###


...

[14] Test Loss: 0.1833	 accuracy: 93.67%

### EPOCH 15 ###


  0%|          | 0/1462 [00:00<?, ?it/s]

Train Epoch: 15 [1600/23389 (7%)]	Loss: 0.009848	 Acc : 0.066
Train Epoch: 15 [3200/23389 (14%)]	Loss: 0.247051	 Acc : 0.133
Train Epoch: 15 [4800/23389 (21%)]	Loss: 0.059638	 Acc : 0.199
Train Epoch: 15 [6400/23389 (27%)]	Loss: 0.571438	 Acc : 0.266
Train Epoch: 15 [8000/23389 (34%)]	Loss: 0.082946	 Acc : 0.333
Train Epoch: 15 [9600/23389 (41%)]	Loss: 0.003343	 Acc : 0.399
Train Epoch: 15 [11200/23389 (48%)]	Loss: 0.020955	 Acc : 0.466
Train Epoch: 15 [12800/23389 (55%)]	Loss: 0.059950	 Acc : 0.533
Train Epoch: 15 [14400/23389 (62%)]	Loss: 0.075334	 Acc : 0.599
Train Epoch: 15 [16000/23389 (68%)]	Loss: 0.012315	 Acc : 0.664
Train Epoch: 15 [17600/23389 (75%)]	Loss: 0.076994	 Acc : 0.731
Train Epoch: 15 [19200/23389 (82%)]	Loss: 0.042618	 Acc : 0.798
Train Epoch: 15 [20800/23389 (89%)]	Loss: 0.041013	 Acc : 0.865
Train Epoch: 15 [22400/23389 (96%)]	Loss: 0.096878	 Acc : 0.932


  0%|          | 0/19 [00:00<?, ?it/s]

[15] Test Loss: 0.0812	 accuracy: 97.67%



In [22]:
result = evaluate(model, valid_loader)
print("Loss : {:.4f}\t acc : {:.2f}".format(result[0], result[1]))

  0%|          | 0/19 [00:00<?, ?it/s]

tensor([1, 1, 1, 2, 0, 1, 1, 0, 1, 1, 0, 2], device='cuda:0')
tensor([0, 1, 1, 2, 0, 2, 1, 0, 1, 1, 0, 2], device='cuda:0')
Loss : 0.1165	 acc : 96.33
